# 02 - Entraînement du CNN (Personne 1)

**Objectif** : construire, entraîner et sauvegarder le CNN (`models/best_model.keras`), puis produire les courbes
Accuracy et Loss.

Le jeu de **test n'est pas utilisé ici** : il est réservé à l'évaluation finale (Personne 2).

In [ ]:
import sys
from pathlib import Path

# permet d'importer le dossier src/ depuis le dossier notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

from src.common import BEST_MODEL_PATH, CLASS_NAMES, CLASS_NAMES_PATH, MODELS_DIR, RESULTS_DIR, SEED
from src.data_loader import load_split
from src.model import build_cnn

tf.keras.utils.set_random_seed(SEED)
MODELS_DIR.mkdir(exist_ok=True); RESULTS_DIR.mkdir(exist_ok=True)
print("TensorFlow", tf.__version__, "| GPU :", tf.config.list_physical_devices("GPU") or "aucun (CPU)")

## 1. Chargement des données

In [ ]:
BATCH_SIZE = 32
train_ds = load_split("train", BATCH_SIZE, shuffle=True)
val_ds = load_split("validation", BATCH_SIZE)

## 2. Architecture du CNN

4 blocs *Convolution → BatchNormalization → ReLU → MaxPooling*, puis `GlobalAveragePooling`,
une couche dense, du Dropout et une sortie softmax à 6 classes. La normalisation (÷255) et l'augmentation
de données (flips, légère rotation) sont intégrées au modèle.

In [ ]:
model = build_cnn(learning_rate=1e-3, dropout=0.4)
model.summary()

## 3. Entraînement

Fonction de perte : `sparse_categorical_crossentropy` · Optimiseur : `Adam`.
Callbacks : sauvegarde du meilleur modèle (selon `val_accuracy`), arrêt anticipé, réduction du taux d'apprentissage.

In [ ]:
EPOCHS = 40

callbacks = [
    keras.callbacks.ModelCheckpoint(BEST_MODEL_PATH, monitor="val_accuracy",
                                    save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

## 4. Courbes Accuracy et Loss

In [ ]:
hist = pd.DataFrame(history.history)
hist.index = np.arange(1, len(hist) + 1)
hist.to_csv(RESULTS_DIR / "history.csv", index_label="epoch")

for metric, title, fname in [("accuracy", "Accuracy", "accuracy.png"), ("loss", "Loss", "loss.png")]:
    plt.figure(figsize=(7, 4))
    plt.plot(hist.index, hist[metric], label="Entraînement")
    plt.plot(hist.index, hist[f"val_{metric}"], label="Validation")
    plt.xlabel("Époque"); plt.ylabel(title); plt.title(f"{title} - entraînement vs validation")
    plt.grid(alpha=0.3); plt.legend(); plt.tight_layout()
    plt.savefig(RESULTS_DIR / fname, dpi=150)
    plt.show()

## 5. Sauvegarde et vérification du modèle

`best_model.keras` a été enregistré par le callback (meilleure `val_accuracy`). On enregistre aussi l'ordre des
classes (`class_names.json`) et on vérifie que le fichier se recharge correctement.

In [ ]:
CLASS_NAMES_PATH.write_text(json.dumps(CLASS_NAMES, ensure_ascii=False, indent=2), encoding="utf-8")

best = keras.models.load_model(BEST_MODEL_PATH)
val_loss, val_acc = best.evaluate(val_ds, verbose=0)
print(f"best_model.keras rechargé -> val_accuracy = {val_acc:.4f} | val_loss = {val_loss:.4f}")
print(f"TensorFlow {tf.__version__} / Keras {keras.__version__}  (à noter dans requirements.txt)")

## 6. Pistes d'amélioration (J8-J9)

Si la validation plafonne : augmenter `EPOCHS`, ajuster `learning_rate` / `dropout`, ajouter des blocs ou des filtres
dans `src/model.py`, renforcer l'augmentation. Relancer ensuite les cellules 2 à 5 et pousser le nouveau
`best_model.keras` sur Git.

*Notes de Personne 1 :* configuration finale retenue, accuracy de validation obtenue, difficultés rencontrées.